## Neural Networks - Deep Learning
**Theodora Tzina - AEM: 10715**
### Exercise 2
#### ***Part A: Support Vector Machines for 2-class classification***

In [ ]:
import time
import dataProccessing as dp
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

**CIFAR-10 dataset preparation**

The dataset was modified from 10 --> 2 super-classes:

In [4]:
# 1. Load CIFAR-10 dataset
x_train_raw, y_train_raw, x_test_raw, y_test_raw = dp.load_cifar10()

# 2. Convert CIFAR-10 labels to binary (Animals vs Vehicles)
y_train_bin, y_test_bin = dp.convert_labels(y_train_raw, y_test_raw)

# 3. Split training data into training and validation sets
x_train_split, y_train_split, x_val_split, y_val_split = dp.split_train_validation(x_train_raw, y_train_bin, val_size=0.4)

# 4. Balance CIFAR-10 dataset by undersampling majority class
x_train_bal, y_train_bal = dp.balance_dataset(x_train_split, y_train_split)

# 5. Flatten and normalize CIFAR-10 images
x_train_fl, x_val_fl, x_test_fl = dp.flatten_and_normalize(x_train_bal, x_val_split, x_test_raw)

# 6. Apply PCA for dimensionality reduction
x_train, x_val, x_test = dp.apply_pca(x_train_fl, x_val_fl, x_test_fl, n_components=0.9)

# 7. Rename variables for clarity
y_train = y_train_bal
y_val = y_val_split
y_test = y_test_bin

Loading CIFAR-10 dataset
Original training data shape: (50000, 32, 32, 3)
Original test data shape: (10000, 32, 32, 3)
Flatten training labels shape: (50000,)
Flatten test labels shape: (10000,)

Converting CIFAR-10 labels to binary (Animals vs Vehicles)
CIFAR-10 binary super-classes:
0: Animals (bird, cat, deer, dog, frog, horse)
1: Vehicles (airplane, automobile, ship, truck)
Sample of converted training labels: [0 1 1 0 1 1 0 0 1 0]
Sample of converted test labels: [0 1 1 1 0 0 1 0 0 1]

Splitting training set into training (60.0%) and validation (40.0%)
New training data shape: (30000, 32, 32, 3)
Validation data shape: (20000, 32, 32, 3)

Balancing CIFAR-10 dataset by undersampling majority class
Class 0 samples before balancing: 18000
Class 1 samples before balancing: 12000
Balanced dataset size: 24000 samples
Class 0 samples: 12000
Class 1 samples: 12000

Flattening and normalizing CIFAR-10 images
Flattened training data shape: (24000, 3072)
Flattened validation data shape: (2000

##### ***Linear Kernel***
Testing different values of 'C' on Linear Kernel SVMs:

In [5]:
# Experiment with 'C' values
c_values = [0.1, 1.0, 10.0]

print("\nRunning Experiment 1: Tuning 'C' for Linear Kernel SVM")
print(f"Values to test: {c_values}")
print("="*70)

best_accuracy = 0
best_c = c_values[0]
y_best_pred = None

for c in c_values:
    print(f"\nTesting Linear Kernel with C={c}")
    start_time = time.time()

    # Train Linear SVM
    svm_linear = SVC(kernel='linear', C=c, random_state=42)
    svm_linear.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate Linear SVM on training set
    y_train_pred = svm_linear.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate Linear SVM on validation set
    y_val_pred = svm_linear.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")

    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_c = c
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best 'C' value: {best_c} with accuracy: {best_accuracy:.2f}")


Running Experiment 1: Tuning 'C' for Linear Kernel SVM
Values to test: [0.1, 1.0, 10.0]

Testing Linear Kernel with C=0.1
Training time: 93.10 seconds
Training Accuracy: 81.30%
Validation Accuracy: 80.84%

Testing Linear Kernel with C=1.0
Training time: 452.63 seconds
Training Accuracy: 81.30%
Validation Accuracy: 80.84%

Testing Linear Kernel with C=10.0
Training time: 3289.07 seconds
Training Accuracy: 81.31%
Validation Accuracy: 80.84%

Best 'C' value: 0.1 with accuracy: 0.81


##### ***Polynomial Kernel***
Testing different values of 'Degree' on Polynomial Kernel SVMs for C=0.1:

In [14]:
# Experiment with 'Degree' values for Polynomial Kernel with fixed C=0.1
degrees = [2, 3, 5]

print(f"\nRunning Experiment 2: Tuning 'Degree' for Polynomial Kernel (C=0.1)")
print(f"Degrees to test: {degrees}")
print("="*70)

best_accuracy = 0
best_degree = degrees[0]
y_best_pred = None

for d in degrees:
    print(f"\nTesting Polynomial Kernel with Degree={d}")
    start_time = time.time()
    
    # Train Polynomial SVM
    svm_poly = SVC(kernel='poly', degree=d, C=0.1, coef0=1, random_state=42)
    svm_poly.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")
    
    # Evaluate Polynomial SVM on training set
    y_train_pred = svm_poly.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate Polynomial SVM on validation set
    y_val_pred = svm_poly.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")
    
    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_degree = d
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best Degree value for C=0.1: {best_degree} with {best_accuracy:.2%} accuracy")


Running Experiment 2: Tuning 'Degree' for Polynomial Kernel (C=0.1)
Degrees to test: [2, 3, 5]

Testing Polynomial Kernel with Degree=2
Training time: 25.36 seconds
Training Accuracy: 86.52%
Validation Accuracy: 85.60%

Testing Polynomial Kernel with Degree=3
Training time: 23.06 seconds
Training Accuracy: 89.32%
Validation Accuracy: 86.98%

Testing Polynomial Kernel with Degree=5
Training time: 56.20 seconds
Training Accuracy: 95.39%
Validation Accuracy: 87.72%

Best Degree value for C=0.1: 3 with 86.98% accuracy


##### ***Polynomial Kernel***
Testing different values of 'Degree' on Polynomial Kernel SVMs for C=1:

In [15]:
# Experiment with 'Degree' values for Polynomial Kernel with fixed C=1.0
degrees = [2, 3]

print(f"\nRunning Experiment 3: Tuning 'Degree' for Polynomial Kernel (C=1.0)")
print(f"Degrees to test: {degrees}")
print("="*70)

best_accuracy = 0
best_degree = degrees[0]
y_best_pred = None

for d in degrees:
    print(f"\nTesting Polynomial Kernel with Degree={d}")
    start_time = time.time()

    # Train Polynomial SVM
    svm_poly = SVC(kernel='poly', degree=d, C=1.0, coef0=1, random_state=42)
    svm_poly.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate Polynomial SVM on training set
    y_train_pred = svm_poly.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate Polynomial SVM on validation set
    y_val_pred = svm_poly.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")
    
    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_degree = d
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best Degree value for C=1.0: {best_degree} with {best_accuracy:.2%} accuracy")


Running Experiment 3: Tuning 'Degree' for Polynomial Kernel (C=1.0)
Degrees to test: [2, 3]

Testing Polynomial Kernel with Degree=2
Training time: 25.90 seconds
Training Accuracy: 89.78%
Validation Accuracy: 87.36%

Testing Polynomial Kernel with Degree=3
Training time: 54.71 seconds
Training Accuracy: 94.26%
Validation Accuracy: 88.08%

Best Degree value for C=1.0: 2 with 87.36% accuracy


##### ***Polynomial Kernel***
Testing different values of 'Degree' on Polynomial Kernel SVMs for C=10:

In [16]:
# Experiment with 'Degree' values for Polynomial Kernel with fixed C=10.0
degrees = [2, 3]

print(f"\nRunning Experiment 4: Tuning 'Degree' for Polynomial Kernel (C=10.0)")
print(f"Degrees to test: {degrees}")
print("="*70)

best_accuracy = 0
best_degree = degrees[0]
y_best_pred = None

for d in degrees:
    print(f"\nTesting Polynomial Kernel with Degree={d}")
    start_time = time.time()

    # Train Polynomial SVM
    svm_poly = SVC(kernel='poly', degree=d, C=10.0, coef0=1, random_state=42)
    svm_poly.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate Polynomial SVM on training set
    y_train_pred = svm_poly.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate Polynomial SVM on validation set
    y_val_pred = svm_poly.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")
    
    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_degree = d
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best Degree value for C=10.0: {best_degree} with {best_accuracy:.2%} accuracy")


Running Experiment 4: Tuning 'Degree' for Polynomial Kernel (C=10.0)
Degrees to test: [2, 3]

Testing Polynomial Kernel with Degree=2
Training time: 91.54 seconds
Training Accuracy: 92.80%
Validation Accuracy: 87.50%

Testing Polynomial Kernel with Degree=3
Training time: 155.09 seconds
Training Accuracy: 98.67%
Validation Accuracy: 86.91%

Best Degree value for C=10.0: 2 with 87.50% accuracy


##### ***RBF Kernel***
Testing different values of 'C' on RBF Kernel SVMs for gamma=0.01:

In [17]:
# Experiment with 'C' values
c_values = [0.1, 1.0, 10.0]

print("\nRunning Experiment 5: Tuning 'C' for RBF Kernel SVM (gamma=0.01)")
print(f"Values to test: {c_values}")
print("="*70)

best_accuracy = 0
best_c = c_values[0]
y_best_pred = None

for c in c_values:
    print(f"\nTesting RBF Kernel with C={c}")
    start_time = time.time()

    # Train RBF SVM
    svm_rbf = SVC(kernel='rbf', C=c, gamma=0.01, random_state=42)
    svm_rbf.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate RBF SVM on training set
    y_train_pred = svm_rbf.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate RBF SVM on validation set
    y_val_pred = svm_rbf.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")

    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_c = c
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best 'C' value for gamma=0.01: {best_c} with accuracy: {best_accuracy:.2%}")


Running Experiment 5: Tuning 'C' for RBF Kernel SVM (gamma=0.01)
Values to test: [0.1, 1.0, 10.0]

Testing RBF Kernel with C=0.1
Training time: 36.32 seconds
Training Accuracy: 86.17%
Validation Accuracy: 83.97%

Testing RBF Kernel with C=1.0
Training time: 54.71 seconds
Training Accuracy: 94.43%
Validation Accuracy: 87.67%

Testing RBF Kernel with C=10.0
Training time: 189.07 seconds
Training Accuracy: 99.62%
Validation Accuracy: 87.77%

Best 'C' value for gamma=0.01: 1.0 with accuracy: 87.67%


##### ***RBF Kernel***
Testing different values of 'C' on RBF Kernel SVMs for gamma=0.001:

In [18]:
# Experiment with 'C' values
c_values = [1.0, 10.0, 100.0]

print("\nRunning Experiment 6: Tuning 'C' for RBF Kernel SVM (gamma=0.001)")
print(f"Values to test: {c_values}")
print("="*70)

best_accuracy = 0
best_c = c_values[0]
y_best_pred = None

for c in c_values:
    print(f"\nTesting RBF Kernel with C={c}")
    start_time = time.time()

    # Train RBF SVM
    svm_rbf = SVC(kernel='rbf', C=c, gamma=0.001, random_state=42)
    svm_rbf.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate RBF SVM on training set
    y_train_pred = svm_rbf.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate RBF SVM on validation set
    y_val_pred = svm_rbf.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")

    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_c = c
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best 'C' value for gamma=0.001: {best_c} with accuracy: {best_accuracy:.2%}")


Running Experiment 6: Tuning 'C' for RBF Kernel SVM (gamma=0.001)
Values to test: [1.0, 10.0, 100.0]

Testing RBF Kernel with C=1.0
Training time: 27.93 seconds
Training Accuracy: 85.53%
Validation Accuracy: 84.62%

Testing RBF Kernel with C=10.0
Training time: 28.30 seconds
Training Accuracy: 89.41%
Validation Accuracy: 87.15%

Testing RBF Kernel with C=100.0
Training time: 109.02 seconds
Training Accuracy: 94.12%
Validation Accuracy: 88.37%

Best 'C' value for gamma=0.001: 100.0 with accuracy: 88.37%


##### ***RBF Kernel***
Testing different values of 'C' on RBF Kernel SVMs for gamma='scale':

In [19]:
# Experiment with 'C' values
c_values = [1.0, 10.0, 100.0]

print("\nRunning Experiment 7: Tuning 'C' for RBF Kernel SVM (gamma='scale')")
print(f"Values to test: {c_values}")
print("="*70)

best_accuracy = 0
best_c = c_values[0]
y_best_pred = None

for c in c_values:
    print(f"\nTesting RBF Kernel with C={c}")
    start_time = time.time()

    # Train RBF SVM
    svm_rbf = SVC(kernel='rbf', C=c, gamma='scale', random_state=42)
    svm_rbf.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate RBF SVM on training set
    y_train_pred = svm_rbf.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate RBF SVM on validation set
    y_val_pred = svm_rbf.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")

    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_c = c
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best 'C' value for gamma='scale': {best_c} with accuracy: {best_accuracy:.2%}")


Running Experiment 7: Tuning 'C' for RBF Kernel SVM (gamma='scale')
Values to test: [1.0, 10.0, 100.0]

Testing RBF Kernel with C=1.0
Training time: 86.32 seconds
Training Accuracy: 91.60%
Validation Accuracy: 87.33%

Testing RBF Kernel with C=10.0
Training time: 1195.68 seconds
Training Accuracy: 98.20%
Validation Accuracy: 88.11%

Testing RBF Kernel with C=100.0
Training time: 540.86 seconds
Training Accuracy: 99.92%
Validation Accuracy: 86.41%

Best 'C' value for gamma='scale': 1.0 with accuracy: 87.33%


##### ***Sigmoid Kernel***
Testing different values of 'C' on Sigmoid Kernel SVMs for gamma='scale':

In [20]:
# Experiment with 'C' values
c_values = [1.0, 10.0]

print("\nRunning Experiment 8: Tuning 'C' for Sigmoid Kernel SVM (gamma='scale')")
print(f"Values to test: {c_values}")
print("="*70)

best_accuracy = 0
best_c = c_values[0]
y_best_pred = None

for c in c_values:
    print(f"\nTesting Sigmoid Kernel with C={c}")
    start_time = time.time()

    # Train Sigmoid SVM
    svm_sigmoid = SVC(kernel='sigmoid', C=c, gamma='scale', coef0=0, random_state=42)
    svm_sigmoid.fit(x_train, y_train)
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.2f} seconds")

    # Evaluate Sigmoid SVM on training set
    y_train_pred = svm_sigmoid.predict(x_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"Training Accuracy: {train_accuracy:.2%}")

    # Evaluate Sigmoid SVM on validation set
    y_val_pred = svm_sigmoid.predict(x_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy: {val_accuracy:.2%}")

    gap = train_accuracy - val_accuracy
    is_better = False
    
    # Rule 1: Disqualify massive overfitting immediately
    if gap > 0.10:
        is_better = False
        
    # Rule 2: If it's the first valid model, take it
    elif best_accuracy == 0:
        is_better = True
        
    # Rule 3: If Accuracy beats previous best by a significant margin (> 1%) we tolerate a slight gap
    elif val_accuracy > (best_accuracy + 0.01):
        is_better = True

    # Rule 4: If Accuracy is similar (within 1%), but has a much tighter gap, take it.
    elif (val_accuracy > best_accuracy - 0.005) and (gap < best_gap - 0.02):
        is_better = True

    # Apply the update
    if is_better:
        best_accuracy = val_accuracy
        best_gap = gap
        best_c = c
        y_best_pred = y_val_pred

print("\n" + "="*70)
print(f"Best 'C' value for gamma='scale': {best_c} with accuracy: {best_accuracy:.2%}")


Running Experiment 8: Tuning 'C' for Sigmoid Kernel SVM (gamma='scale')
Values to test: [1.0, 10.0]

Testing Sigmoid Kernel with C=1.0
Training time: 61.23 seconds
Training Accuracy: 64.14%
Validation Accuracy: 63.79%

Testing Sigmoid Kernel with C=10.0
Training time: 63.25 seconds
Training Accuracy: 64.08%
Validation Accuracy: 63.73%

Best 'C' value for gamma='scale': 1.0 with accuracy: 63.79%
